In [ ]:
"'Model 4: iTransformer'"

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# ── Paths ─────────────────────────────────────────────────────
DRIVE_DIR  = '/content/drive/MyDrive/iTransformer'
OUTPUT_DIR = f'{DRIVE_DIR}/model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Add Drive folder to Python path so data_loader.py is importable
sys.path.insert(0, DRIVE_DIR)

# ── Imports ───────────────────────────────────────────────────
import time
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Device ────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM: {total_mem:.1f} GB")
else:
    print("  ⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS     = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']
CLIP_RANGES = [(2,8), (5,200), (1,7), (1,7)]

# Quantile outputs — P10, P50, P90 per target
# P90 used as final threshold prediction
QUANTILES = [0.10, 0.50, 0.90]

EPOCHS      = 60
BATCH_SIZE  = 1024   # T4 16GB — safe and fast (use 2048 for A100)
LR          = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE    = 10
GRAD_CLIP   = 1.0

# ── Load data from Google Drive ──────────────────────────────
from data_loader import load_data

# data_loader expects X_train.csv / X_test.csv in the data_dir
# We uploaded them as X_train_enriched.csv / X_test_enriched.csv
# Temporarily rename so data_loader finds them
import shutil
shutil.copy(f'{DRIVE_DIR}/X_train_enriched.csv', f'{DRIVE_DIR}/X_train.csv')
shutil.copy(f'{DRIVE_DIR}/X_test_enriched.csv',  f'{DRIVE_DIR}/X_test.csv')

X_train, X_test, y_train, y_test, FEAT_COLS, X_train_df, X_test_df = load_data(
    data_dir=DRIVE_DIR
)
n_features = len(FEAT_COLS)

# ── Dataset ───────────────────────────────────────────────────
class iTransformerDataset(Dataset):
    def __init__(self, X_df, y_df):
        self.X = torch.tensor(
            X_df[FEAT_COLS].values.astype(np.float32))
        self.y = torch.tensor(
            y_df[TARGET_COLS].values.astype(np.float32))

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# Load y files from Drive
y_train_df = pd.read_csv(f'{DRIVE_DIR}/y_train.csv')
y_test_df  = pd.read_csv(f'{DRIVE_DIR}/y_test.csv')

train_ds = iTransformerDataset(X_train_df, y_train_df)
test_ds  = iTransformerDataset(X_test_df,  y_test_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2,
                          shuffle=False, num_workers=2, pin_memory=True)

# ── iTransformer Architecture ─────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class iTransformerLayer(nn.Module):
    """
    Single iTransformer layer.
    Input:  (B, n_features, d_model)  — each feature is a token
    Output: (B, n_features, d_model)

    Attention is computed ACROSS FEATURES (not across time steps).
    This allows the model to learn which features are correlated
    and how they jointly determine thresholds.
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # Pre-norm architecture (more stable than post-norm)
        # x: (B, n_features, d_model)
        residual = x
        x = self.norm1(x)
        # Self-attention ACROSS FEATURES (inverted)
        x, _ = self.attn(x, x, x)
        x = self.drop(x) + residual

        residual = x
        x = self.norm2(x)
        x = self.ff(x) + residual
        return x


class iTransformer(nn.Module):
    """
    Full iTransformer for multi-output regression.

    Following the original paper:
    - Each feature is projected to d_model dimension
    - Inverted attention: features attend to each other
    - Global pooling produces final representation
    - Multiple quantile heads for uncertainty quantification
    """

    def __init__(self, n_features, d_model=128, n_heads=8,
                 n_layers=4, d_ff=512, dropout=0.1,
                 output_dim=4, n_quantiles=3):
        super().__init__()
        self.n_features  = n_features
        self.d_model     = d_model
        self.n_quantiles = n_quantiles
        self.output_dim  = output_dim

        # ── Feature Embedding ─────────────────────────────────
        # Each feature (scalar) projected to d_model
        # This is the "inverted" embedding — one per feature, not per time step
        self.feature_embedding = nn.Linear(1, d_model)

        # Learnable feature position encoding
        # Each feature gets its own position token (not temporal)
        self.feature_pos_enc = nn.Parameter(
            torch.randn(1, n_features, d_model) * 0.02
        )

        # ── iTransformer Encoder ──────────────────────────────
        self.encoder = nn.ModuleList([
            iTransformerLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model)

        # ── Projection Head ───────────────────────────────────
        # Global average pool + projection
        self.pre_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.LayerNorm(d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Quantile output heads (one per quantile per target)
        # Each head predicts all targets for one quantile
        self.quantile_heads = nn.ModuleList([
            nn.Linear(d_model // 2, output_dim)
            for _ in range(n_quantiles)
        ])

        # Weight initialization
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        x: (B, n_features)  — flat feature vector
        Returns: list of (B, output_dim) for each quantile
        """
        B = x.size(0)

        # Embed: (B, n_features) → (B, n_features, d_model)
        # Each feature value is treated as a single-step "series"
        x_emb = self.feature_embedding(x.unsqueeze(-1))

        # Add feature positional encoding
        x_emb = x_emb + self.feature_pos_enc

        # iTransformer encoder: attention across features
        for layer in self.encoder:
            x_emb = layer(x_emb)
        x_emb = self.encoder_norm(x_emb)

        # Global average pool across feature dimension
        # (B, n_features, d_model) → (B, d_model)
        pooled = x_emb.mean(dim=1)

        # Projection
        h = self.pre_head(pooled)  # (B, d_model//2)

        # Quantile predictions
        quantile_preds = [head(h) for head in self.quantile_heads]
        return quantile_preds  # list of 3 × (B, 4)


# ── Combined Quantile Loss ────────────────────────────────────
class MultiQuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, preds, y_true):
        """
        preds: list of (B, output_dim) tensors
        y_true: (B, output_dim)
        """
        total_loss = 0.0
        for q, pred in zip(self.quantiles, preds):
            diff = y_true - pred
            loss = torch.where(diff >= 0, q * diff, (q-1) * diff)
            total_loss += loss.mean()
        return total_loss / len(self.quantiles)


# ── Model init ────────────────────────────────────────────────
model = iTransformer(
    n_features=n_features,
    d_model=128,
    n_heads=8,
    n_layers=4,
    d_ff=512,
    dropout=0.1,
    output_dim=4,
    n_quantiles=len(QUANTILES),
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters: {total_params:,}")
print(f"Architecture: {n_features} feature tokens × 128 d_model × 4 layers × 8 heads")

criterion = MultiQuantileLoss(QUANTILES)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    anneal_strategy='cos',
)

# ── Training ──────────────────────────────────────────────────
print(f"\nTraining iTransformer for {EPOCHS} epochs...")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'LR':>12}")
print("-" * 48)

best_val   = float('inf')
patience   = 0
best_state = None
train_losses, val_losses = [], []
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X)
        loss  = criterion(preds, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            preds = model(X)
            val_loss += criterion(preds, y).item()
    val_loss /= len(test_loader)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  {epoch:4d}   {tr_loss:>10.4f}   {val_loss:>8.4f}   {lr_now:>10.6f}")

    if val_loss < best_val:
        best_val   = val_loss
        patience   = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch}")
            break

train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f}s ({train_time/60:.1f} min)")

model.load_state_dict(best_state)

# ── Evaluate ──────────────────────────────────────────────────
print("\nEvaluating...")
model.eval()
all_q10, all_q50, all_q90 = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(DEVICE)
        preds = model(X)
        all_q10.append(preds[0].cpu().numpy())
        all_q50.append(preds[1].cpu().numpy())
        all_q90.append(preds[2].cpu().numpy())

y_q10  = np.vstack(all_q10)
y_q50  = np.vstack(all_q50)
y_q90  = np.vstack(all_q90)
y_true = y_test_df[TARGET_COLS].values.astype(np.float32)

# Use P90 as threshold prediction (conservative — flags more cells)
y_pred = y_q90.copy()
for i, (lo, hi) in enumerate(CLIP_RANGES):
    y_pred[:, i] = np.clip(y_pred[:, i], lo, hi)

print(f"\n{'Target':<35} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 62)
results = {}
all_mae, all_rmse, all_r2 = [], [], []
for i, col in enumerate(TARGET_COLS):
    mae  = mean_absolute_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
    r2   = r2_score(y_true[:, i], y_pred[:, i])
    short = col.replace('TARGET_', '')
    print(f"  {short:<33} {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}")
    results[col] = {'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)}
    all_mae.append(mae); all_rmse.append(rmse); all_r2.append(r2)

print("-" * 62)
print(f"  {'AVERAGE':<33} "
      f"{np.mean(all_mae):>8.4f} "
      f"{np.mean(all_rmse):>8.4f} "
      f"{np.mean(all_r2):>8.4f}")

# ── Save ──────────────────────────────────────────────────────
torch.save({
    'model_state_dict': best_state,
    'model_config': {
        'n_features': n_features,
        'd_model': 128,
        'n_heads': 8,
        'n_layers': 4,
        'd_ff': 512,
        'dropout': 0.1,
        'output_dim': 4,
        'n_quantiles': len(QUANTILES),
    },
    'feat_cols':   FEAT_COLS,
    'target_cols': TARGET_COLS,
    'quantiles':   QUANTILES,
}, f'{OUTPUT_DIR}/model4_itransformer.pt')

# Save P90 predictions (threshold)
pd.DataFrame(y_pred, columns=[c+'_pred' for c in TARGET_COLS]).to_csv(
    f'{OUTPUT_DIR}/model4_predictions.csv', index=False)

# Save all quantile predictions (for uncertainty analysis in paper)
all_quant = np.hstack([y_q10, y_q50, y_q90])
quant_cols = (
    [c+'_q10' for c in TARGET_COLS] +
    [c+'_q50' for c in TARGET_COLS] +
    [c+'_q90' for c in TARGET_COLS]
)
pd.DataFrame(all_quant, columns=quant_cols).to_csv(
    f'{OUTPUT_DIR}/model4_quantile_predictions.csv', index=False)

summary = {
    'model': 'iTransformer_2024ICLR',
    'paper': 'Liu et al., ICLR 2024 — iTransformer',
    'train_time_sec': round(train_time, 2),
    'total_params': total_params,
    'architecture': {
        'n_features_as_tokens': n_features,
        'd_model': 128,
        'n_heads': 8,
        'n_layers': 4,
        'd_ff': 512,
        'attention_type': 'inverted (across features, not time)',
    },
    'best_val_loss': round(best_val, 4),
    'avg_MAE':  round(np.mean(all_mae), 4),
    'avg_RMSE': round(np.mean(all_rmse), 4),
    'avg_R2':   round(np.mean(all_r2), 4),
    'per_target': results,
    'quantile_used_for_threshold': 'P90',
    'train_losses': train_losses,
    'val_losses':   val_losses,
}
with open(f'{OUTPUT_DIR}/model4_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Model 4 complete. All outputs saved to Google Drive:")
print(f"   {OUTPUT_DIR}/model4_itransformer.pt")
print(f"   {OUTPUT_DIR}/model4_predictions.csv")
print(f"   {OUTPUT_DIR}/model4_quantile_predictions.csv")
print(f"   {OUTPUT_DIR}/model4_summary.json")
print(f"\n   Download these 4 files to your local model_outputs/ folder")
print(f"   then run evaluate_all_models.py locally.")

Mounted at /content/drive
Using device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB
Loading data from /content/drive/MyDrive/iTransformer...
  Imputing 4 columns with NaN values...
  X_train: (201984, 123), y_train: (201984, 4)
  X_test:  (135120, 123),  y_test:  (135120, 4)
  Features: 123
  NaN check: 0 train, 0 test

Model parameters: 818,508
Architecture: 123 feature tokens × 128 d_model × 4 layers × 8 heads

Training iTransformer for 60 epochs...
 Epoch   Train Loss   Val Loss           LR
------------------------------------------------
     1       0.9231     0.4713     0.000104
     5       0.5099     0.2385     0.000936
    10       0.0766     0.1259     0.000986
    15       0.0628     0.1234     0.000933
    20       0.0575     0.1174     0.000843
    25       0.0527     0.1131     0.000724
    30       0.0499     0.1138     0.000587
    35       0.0470     0.1142     0.000442

  Early stopping at epoch 35

Training time: 5124.6s (85.4 min)

Evaluating...

Target                   

In [ ]:
"""
=============================================================
MODEL 1 — XGBOOST BASELINE (Google Colab Version)
=============================================================
Self-contained — no external dependencies except xgboost.
Loads data directly from Google Drive.

Files needed in /content/drive/MyDrive/iTransformer/:
  X_train_enriched.csv
  X_test_enriched.csv
  y_train.csv
  y_test.csv

Output saved to:
  /content/drive/MyDrive/iTransformer/model_outputs/
    model1_xgboost_predictions.csv
    model1_xgboost_summary.json

Runtime: CPU is fine — expects ~3-5 minutes total.
=============================================================

# ── COLAB SETUP CELL (run this first) ──────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install xgboost scikit-learn pandas numpy -q
"""

# ── Imports ───────────────────────────────────────────────────
import os
import sys
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

# Mount drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Paths ─────────────────────────────────────────────────────
DRIVE_DIR  = '/content/drive/MyDrive/iTransformer'
OUTPUT_DIR = f'{DRIVE_DIR}/model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS     = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']
CLIP_RANGES = [(2, 8), (5, 200), (1, 7), (1, 7)]

# ── Robust data loader (inline — no external data_loader.py needed) ──
def load_clean_data(drive_dir):
    print("Loading data from Google Drive...")

    X_train_df = pd.read_csv(f'{drive_dir}/X_train_enriched.csv', low_memory=False)
    X_test_df  = pd.read_csv(f'{drive_dir}/X_test_enriched.csv',  low_memory=False)
    y_train_df = pd.read_csv(f'{drive_dir}/y_train.csv',          low_memory=False)
    y_test_df  = pd.read_csv(f'{drive_dir}/y_test.csv',           low_memory=False)

    FEAT_COLS = [c for c in X_train_df.columns if c not in ID_COLS]

    # Convert bool columns to int
    for df in [X_train_df, X_test_df]:
        for col in FEAT_COLS:
            if df[col].dtype == bool:
                df[col] = df[col].astype(int)
            elif df[col].dtype == object:
                unique_vals = set(df[col].dropna().unique())
                if unique_vals <= {'True', 'False'}:
                    df[col] = df[col].map({'True': 1, 'False': 0}).astype(float)

    # Impute NaN with train median
    for col in FEAT_COLS:
        if X_train_df[col].isnull().any():
            median_val = X_train_df[col].median()
            X_train_df[col] = X_train_df[col].fillna(median_val)
            X_test_df[col]  = X_test_df[col].fillna(median_val)

    X_train = X_train_df[FEAT_COLS].values.astype(np.float32)
    X_test  = X_test_df[FEAT_COLS].values.astype(np.float32)
    y_train = y_train_df[TARGET_COLS].values.astype(np.float32)
    y_test  = y_test_df[TARGET_COLS].values.astype(np.float32)

    print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"  X_test:  {X_test.shape},  y_test:  {y_test.shape}")
    print(f"  Features: {len(FEAT_COLS)}")
    print(f"  NaN check — train: {np.isnan(X_train).sum()}, test: {np.isnan(X_test).sum()}")

    return X_train, X_test, y_train, y_test, FEAT_COLS, X_train_df, X_test_df, y_test_df

X_train, X_test, y_train, y_test, FEAT_COLS, X_train_df, X_test_df, y_test_df = \
    load_clean_data(DRIVE_DIR)

# ── XGBoost training ──────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    print("\nXGBoost available ✅")
except ImportError:
    print("Installing XGBoost...")
    os.system("pip install xgboost -q")
    from xgboost import XGBRegressor

# Per-target parameters
# Each target has different characteristics so we tune separately
PARAMS = {
    'TARGET_threshold_hours': dict(
        n_estimators     = 500,
        max_depth        = 6,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 20,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        objective        = 'reg:pseudohubererror',
        tree_method      = 'hist',   # fast on CPU
        random_state     = 42,
        n_jobs           = -1,
    ),
    'TARGET_threshold_inactive_min': dict(
        n_estimators     = 500,
        max_depth        = 7,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.7,
        min_child_weight = 10,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        objective        = 'reg:pseudohubererror',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    ),
    'TARGET_threshold_fluctuation': dict(
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 30,
        reg_alpha        = 0.5,
        reg_lambda       = 1.0,
        objective        = 'reg:pseudohubererror',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    ),
    'TARGET_each_hour_fluctuation': dict(
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 30,
        reg_alpha        = 0.5,
        reg_lambda       = 1.0,
        objective        = 'reg:pseudohubererror',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    ),
}

# ── Important: restart runtime before running if you reuploaded data files ──
# Runtime → Restart session → Run all cells again
# This ensures no stale data remains in memory

print(f"Data shapes confirmed:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape},  y_test:  {y_test.shape}")
assert X_train.shape[0] == y_train.shape[0], "Train size mismatch!"
assert X_test.shape[0]  == y_test.shape[0],  "Test size mismatch!"
print("  ✅ Shapes validated")

print(f"\nTraining XGBoost — 4 separate models (one per target)...")
print(f"{'Target':<35} {'Time':>8} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 65)

models    = {}
results   = {}
all_preds = np.zeros_like(y_test)
t_total   = time.time()

for i, target in enumerate(TARGET_COLS):
    t0    = time.time()
    model = XGBRegressor(**PARAMS[target])

    model.fit(X_train, y_train[:, i], verbose=False)

    pred = model.predict(X_test)
    lo, hi = CLIP_RANGES[i]
    pred = np.clip(pred, lo, hi)
    all_preds[:, i] = pred

    mae     = mean_absolute_error(y_test[:, i], pred)
    rmse    = np.sqrt(mean_squared_error(y_test[:, i], pred))
    r2      = r2_score(y_test[:, i], pred)
    elapsed = time.time() - t0

    short = target.replace('TARGET_', '')
    print(f"  {short:<33} {elapsed:>6.1f}s {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}")

    models[target]  = model
    results[target] = {
        'MAE':  round(mae,  4),
        'RMSE': round(rmse, 4),
        'R2':   round(r2,   4),
    }

total_time = time.time() - t_total
avg_mae    = np.mean([v['MAE']  for v in results.values()])
avg_rmse   = np.mean([v['RMSE'] for v in results.values()])
avg_r2     = np.mean([v['R2']   for v in results.values()])

print("-" * 65)
print(f"  {'AVERAGE':<33} {total_time:>6.1f}s "
      f"{avg_mae:>8.4f} {avg_rmse:>8.4f} {avg_r2:>8.4f}")
print(f"\nTotal training time: {total_time/60:.1f} minutes")

# ── Feature importances ───────────────────────────────────────
print("\nTop 15 features (average importance across all targets):")
importances = np.mean([
    models[t].feature_importances_ for t in TARGET_COLS
], axis=0)
top_idx = np.argsort(importances)[::-1][:15]
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {FEAT_COLS[idx]:<45} {importances[idx]:.4f}")

# ── Statistical significance vs other models ──────────────────
print("\nNote: Wilcoxon test vs other models requires their prediction files.")
print("Run evaluate_all_models.py locally after downloading this output.")

# ── Save predictions ──────────────────────────────────────────
pred_df = pd.DataFrame(
    all_preds,
    columns=[c + '_pred' for c in TARGET_COLS]
)
pred_df.to_csv(f'{OUTPUT_DIR}/model1_predictions.csv', index=False)

# ── Save summary ──────────────────────────────────────────────
summary = {
    'model':          'XGBoost_Baseline',
    'library':        'xgboost',
    'train_time_sec': round(total_time, 2),
    'avg_MAE':        round(avg_mae,  4),
    'avg_RMSE':       round(avg_rmse, 4),
    'avg_R2':         round(avg_r2,   4),
    'per_target':     results,
    'top_features': [
        (FEAT_COLS[i], round(float(importances[i]), 4))
        for i in top_idx[:15]
    ],
    'params': {t: {k: v for k, v in p.items() if k != 'n_jobs'}
               for t, p in PARAMS.items()},
}
with open(f'{OUTPUT_DIR}/model1_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Saved to Google Drive:")
print(f"   {OUTPUT_DIR}/model1_predictions.csv")
print(f"   {OUTPUT_DIR}/model1_summary.json")
print(f"\nDownload these two files and share them here.")

Mounted at /content/drive
Loading data from Google Drive...
  X_train: (201984, 123), y_train: (201984, 4)
  X_test:  (135120, 123),  y_test:  (135120, 4)
  Features: 123
  NaN check — train: 0, test: 0

XGBoost available ✅
Data shapes confirmed:
  X_train: (201984, 123), y_train: (201984, 4)
  X_test:  (135120, 123),  y_test:  (135120, 4)
  ✅ Shapes validated

Training XGBoost — 4 separate models (one per target)...
Target                                  Time      MAE     RMSE       R²
-----------------------------------------------------------------
  threshold_hours                     19.2s   1.2034   2.5522 -12.5033
  threshold_inactive_min              13.5s 143.3483 143.9753 -113.4414
  threshold_fluctuation               10.3s   0.0135   0.0837   0.8970
  each_hour_fluctuation               11.0s   0.0135   0.0837   0.8970
-----------------------------------------------------------------
  AVERAGE                             54.0s  36.1447  36.6737 -31.0377

Total training tim

In [ ]:
"""
=============================================================
MODEL 1 — XGBoost — COLAB VERSION (Self-Contained)
=============================================================
NO external dependencies. No data_loader.py needed.
All paths are explicit. Debug output at every step.

Instructions:
  1. Runtime → Change runtime type → T4 GPU (or CPU is fine)
  2. Make sure these files are in your Drive:
       /content/drive/MyDrive/iTransformer/X_train_enriched.csv
       /content/drive/MyDrive/iTransformer/X_test_enriched.csv
       /content/drive/MyDrive/iTransformer/y_train.csv
       /content/drive/MyDrive/iTransformer/y_test.csv
  3. Run this cell:
       exec(open('/content/drive/MyDrive/iTransformer/train_model1_xgboost.py').read())
=============================================================
"""

# ── Step 1: Mount Drive ───────────────────────────────────────
print("Step 1: Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("  Done")

# ── Step 2: Imports ───────────────────────────────────────────
print("\nStep 2: Installing xgboost...")
import subprocess
subprocess.run(["pip", "install", "xgboost", "-q"])

import os, time, json, pickle
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print("  Done")

# ── Step 3: Paths ─────────────────────────────────────────────
DRIVE_DIR  = '/content/drive/MyDrive/iTransformer'
OUTPUT_DIR = f'{DRIVE_DIR}/model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nStep 3: Checking files...")
files_needed = {
    'X_train_enriched.csv': 201984,
    'X_test_enriched.csv':  135120,
    'y_train.csv':          201984,
    'y_test.csv':           135120,
}
for fname, expected_rows in files_needed.items():
    path = f'{DRIVE_DIR}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"  {fname}: found ({size:.1f} MB)")
    else:
        raise FileNotFoundError(
            f"\n\n❌ FILE NOT FOUND: {path}\n"
            f"Please upload {fname} to your Drive folder and try again.\n"
        )

# ── Step 4: Load data ─────────────────────────────────────────
print("\nStep 4: Loading data...")

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']
CLIP_RANGES = [(2, 8), (5, 200), (1, 7), (1, 7)]

X_train_df = pd.read_csv(f'{DRIVE_DIR}/X_train_enriched.csv', low_memory=False)
X_test_df  = pd.read_csv(f'{DRIVE_DIR}/X_test_enriched.csv',  low_memory=False)

print(f"  X_train shape: {X_train_df.shape}")
print(f"  X_test  shape: {X_test_df.shape}")

# Strict row count check
if len(X_train_df) != 201984:
    raise ValueError(
        f"\n\n❌ WRONG X_train FILE!\n"
        f"Got {len(X_train_df)} rows but expected 201984.\n"
        f"You uploaded the wrong file. Use X_train_ENRICHED.csv.\n"
    )
if len(X_test_df) != 135120:
    raise ValueError(
        f"\n\n❌ WRONG X_test FILE!\n"
        f"Got {len(X_test_df)} rows but expected 135120.\n"
        f"You uploaded the wrong file. Use X_test_ENRICHED.csv.\n"
    )

# Fix bool columns
for df in [X_train_df, X_test_df]:
    for col in df.columns:
        if col in ID_COLS:
            continue
        if df[col].dtype == bool:
            df[col] = df[col].astype(int)
        elif df[col].dtype == object:
            unique = set(df[col].dropna().astype(str).unique())
            if unique <= {'True', 'False'}:
                df[col] = df[col].map({'True': 1, 'False': 0}).astype(int)

FEAT_COLS = [c for c in X_train_df.columns if c not in ID_COLS]
print(f"  Feature columns: {len(FEAT_COLS)}")

# Impute NaN
for col in FEAT_COLS:
    if X_train_df[col].isnull().any():
        m = X_train_df[col].median()
        X_train_df[col] = X_train_df[col].fillna(m)
        X_test_df[col]  = X_test_df[col].fillna(m)

X_train = X_train_df[FEAT_COLS].values.astype(np.float32)
X_test  = X_test_df[FEAT_COLS].values.astype(np.float32)

print(f"  X_train dtype: {X_train.dtype}")
print(f"  Any NaN in X_train: {np.isnan(X_train).any()}")
print(f"  Any NaN in X_test:  {np.isnan(X_test).any()}")
print(f"  All finite X_train: {np.isfinite(X_train).all()}")

# Load y
y_train_df = pd.read_csv(f'{DRIVE_DIR}/y_train.csv')
y_test_df  = pd.read_csv(f'{DRIVE_DIR}/y_test.csv')

if len(y_train_df) != 201984:
    raise ValueError(f"❌ WRONG y_train! Got {len(y_train_df)} rows, expected 201984")
if len(y_test_df) != 135120:
    raise ValueError(f"❌ WRONG y_test! Got {len(y_test_df)} rows, expected 135120")

y_train = y_train_df[TARGET_COLS].values.astype(np.float32)
y_test  = y_test_df[TARGET_COLS].values.astype(np.float32)

print(f"  y_train shape: {y_train.shape}")
print(f"  y_test  shape: {y_test.shape}")

print("\n  Target distributions (train):")
for i, col in enumerate(TARGET_COLS):
    short = col.replace('TARGET_', '')
    print(f"    {short:<35} "
          f"mean={y_train[:,i].mean():.3f}  "
          f"std={y_train[:,i].std():.3f}  "
          f"max={y_train[:,i].max():.3f}")

print("\n  Target distributions (test):")
for i, col in enumerate(TARGET_COLS):
    short = col.replace('TARGET_', '')
    print(f"    {short:<35} "
          f"mean={y_test[:,i].mean():.3f}  "
          f"std={y_test[:,i].std():.3f}  "
          f"max={y_test[:,i].max():.3f}")

# ── Step 5: Train ─────────────────────────────────────────────
PARAMS = {
    'TARGET_threshold_hours': dict(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
    'TARGET_threshold_inactive_min': dict(
        n_estimators=300, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
    'TARGET_threshold_fluctuation': dict(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=30,
        reg_alpha=0.5, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
    'TARGET_each_hour_fluctuation': dict(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=30,
        reg_alpha=0.5, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
}

print(f"\nStep 5: Training XGBoost — 4 models...")
print(f"{'Target':<35} {'Time':>10} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 72)

models    = {}
results   = {}
all_preds = np.zeros_like(y_test)
t_total   = time.time()

for i, target in enumerate(TARGET_COLS):
    lo, hi = CLIP_RANGES[i]
    t0     = time.time()

    model  = XGBRegressor(**PARAMS[target])
    model.fit(X_train, y_train[:, i], verbose=False)
    pred   = np.clip(model.predict(X_test), lo, hi)
    all_preds[:, i] = pred

    mae  = mean_absolute_error(y_test[:, i], pred)
    rmse = np.sqrt(mean_squared_error(y_test[:, i], pred))
    r2   = r2_score(y_test[:, i], pred)
    t    = time.time() - t0

    short = target.replace('TARGET_', '')
    print(f"  {short:<33} {t:>8.1f}s {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}")
    models[target]  = model
    results[target] = {'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)}

total_time = time.time() - t_total
avg_mae  = np.mean([v['MAE']  for v in results.values()])
avg_rmse = np.mean([v['RMSE'] for v in results.values()])
avg_r2   = np.mean([v['R2']   for v in results.values()])
print("-" * 72)
print(f"  {'AVERAGE':<33} {total_time:>8.1f}s "
      f"{avg_mae:>8.4f} {avg_rmse:>8.4f} {avg_r2:>8.4f}")

# ── Step 6: Feature importances ───────────────────────────────
importances = np.mean([models[t].feature_importances_ for t in TARGET_COLS], axis=0)
top_idx = np.argsort(importances)[::-1][:15]
print(f"\nTop 15 features:")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {FEAT_COLS[idx]:<45} {importances[idx]:.4f}")

# ── Step 7: Save ──────────────────────────────────────────────
print(f"\nStep 7: Saving to {OUTPUT_DIR}...")

pd.DataFrame(all_preds, columns=[c+'_pred' for c in TARGET_COLS]).to_csv(
    f'{OUTPUT_DIR}/model1_predictions.csv', index=False)

with open(f'{OUTPUT_DIR}/model1_summary.json', 'w') as f:
    json.dump({
        'model': 'XGBoost_Baseline',
        'train_time_sec': round(total_time, 2),
        'avg_MAE': round(avg_mae, 4),
        'avg_RMSE': round(avg_rmse, 4),
        'avg_R2': round(avg_r2, 4),
        'per_target': results,
        'top_features': [(FEAT_COLS[i], round(float(importances[i]),4)) for i in top_idx],
        'n_train': int(X_train.shape[0]),
        'n_test':  int(X_test.shape[0]),
        'n_features': int(X_train.shape[1]),
        'train_losses': [],
        'val_losses':   [],
    }, f, indent=2)

with open(f'{OUTPUT_DIR}/model1_xgboost.pkl', 'wb') as f:
    pickle.dump(models, f)

print(f"  ✅ model1_predictions.csv saved")
print(f"  ✅ model1_summary.json saved")
print(f"  ✅ model1_xgboost.pkl saved")
print(f"\n✅ Model 1 complete! Download model1_predictions.csv and model1_summary.json")

Step 1: Mounting Google Drive...
Mounted at /content/drive
  Done

Step 2: Installing xgboost...
  Done

Step 3: Checking files...
  X_train_enriched.csv: found (397.9 MB)
  X_test_enriched.csv: found (264.7 MB)
  y_train.csv: found (8.7 MB)
  y_test.csv: found (5.8 MB)

Step 4: Loading data...
  X_train shape: (201984, 126)
  X_test  shape: (135120, 126)
  Feature columns: 123
  X_train dtype: float32
  Any NaN in X_train: False
  Any NaN in X_test:  False
  All finite X_train: True
  y_train shape: (201984, 4)
  y_test  shape: (135120, 4)

  Target distributions (train):
    threshold_hours                     mean=2.396  std=1.083  max=8.000
    threshold_inactive_min              mean=10.175  std=15.815  max=80.070
    threshold_fluctuation               mean=1.006  std=0.088  max=6.400
    each_hour_fluctuation               mean=1.006  std=0.088  max=6.400

  Target distributions (test):
    threshold_hours                     mean=2.168  std=0.695  max=8.000
    threshold_inacti

In [ ]:
# ============================================================
# PASTE THIS ENTIRE BLOCK INTO A SINGLE COLAB CELL AND RUN IT
# Do NOT use exec() or load from file — paste directly
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, time, json, pickle
import numpy as np
import pandas as pd
import subprocess
subprocess.run(["pip", "install", "xgboost", "-q"])
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DRIVE_DIR  = '/content/drive/MyDrive/iTransformer'
OUTPUT_DIR = f'{DRIVE_DIR}/model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS     = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']
CLIP_RANGES = [(2,8), (5,200), (1,7), (1,7)]

# ── Load ──────────────────────────────────────────────────────
print("Loading data...")
X_train_df = pd.read_csv(f'{DRIVE_DIR}/X_train_enriched.csv', low_memory=False)
X_test_df  = pd.read_csv(f'{DRIVE_DIR}/X_test_enriched.csv',  low_memory=False)
y_train_df = pd.read_csv(f'{DRIVE_DIR}/y_train.csv')
y_test_df  = pd.read_csv(f'{DRIVE_DIR}/y_test.csv')

assert len(X_train_df) == 201984, f"Wrong X_train rows: {len(X_train_df)}"
assert len(X_test_df)  == 135120, f"Wrong X_test rows: {len(X_test_df)}"
assert len(y_train_df) == 201984, f"Wrong y_train rows: {len(y_train_df)}"
assert len(y_test_df)  == 135120, f"Wrong y_test rows: {len(y_test_df)}"

# Fix bool columns
for df in [X_train_df, X_test_df]:
    for col in df.columns:
        if col in ID_COLS: continue
        if df[col].dtype == bool:
            df[col] = df[col].astype(int)
        elif df[col].dtype == object:
            u = set(df[col].dropna().astype(str).unique())
            if u <= {'True','False'}:
                df[col] = df[col].map({'True':1,'False':0}).astype(int)

FEAT_COLS = [c for c in X_train_df.columns if c not in ID_COLS]

# Impute NaN
for col in FEAT_COLS:
    if X_train_df[col].isnull().any():
        m = X_train_df[col].median()
        X_train_df[col] = X_train_df[col].fillna(m)
        X_test_df[col]  = X_test_df[col].fillna(m)

X_train = X_train_df[FEAT_COLS].values.astype(np.float32)
X_test  = X_test_df[FEAT_COLS].values.astype(np.float32)
y_train = y_train_df[TARGET_COLS].values.astype(np.float32)
y_test  = y_test_df[TARGET_COLS].values.astype(np.float32)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

# ── Cap test features to train range ─────────────────────────
print("\nCapping test features to train range...")
train_mins = X_train.min(axis=0)
train_maxs = X_train.max(axis=0)
X_test_capped = X_test.copy()
n_capped = 0
for i in range(X_train.shape[1]):
    before = X_test_capped[:, i].copy()
    X_test_capped[:, i] = np.clip(X_test_capped[:, i], train_mins[i], train_maxs[i])
    if not np.array_equal(before, X_test_capped[:, i]):
        n_capped += 1
print(f"  {n_capped} columns had out-of-range values — clipped to train range")
print(f"  Verification — Window_Inactive_Sum max before: {X_test[:, FEAT_COLS.index('Window_Inactive_Sum')].max():.3f}")
print(f"  Verification — Window_Inactive_Sum max after:  {X_test_capped[:, FEAT_COLS.index('Window_Inactive_Sum')].max():.3f}")

# ── Train ─────────────────────────────────────────────────────
PARAMS = {
    'TARGET_threshold_hours': dict(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
    'TARGET_threshold_inactive_min': dict(
        n_estimators=300, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
    'TARGET_threshold_fluctuation': dict(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=30,
        reg_alpha=0.5, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
    'TARGET_each_hour_fluctuation': dict(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=30,
        reg_alpha=0.5, reg_lambda=1.0,
        objective='reg:pseudohubererror',
        tree_method='hist', random_state=42, n_jobs=-1,
    ),
}

print(f"\nTraining XGBoost (4 models)...")
print(f"{'Target':<35} {'Time':>10} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 72)

models    = {}
results   = {}
all_preds = np.zeros_like(y_test)
t_total   = time.time()

for i, target in enumerate(TARGET_COLS):
    lo, hi = CLIP_RANGES[i]
    t0 = time.time()
    model = XGBRegressor(**PARAMS[target])
    model.fit(X_train, y_train[:, i], verbose=False)

    # Use CAPPED X_test for prediction
    pred = np.clip(model.predict(X_test_capped), lo, hi)
    all_preds[:, i] = pred

    mae  = mean_absolute_error(y_test[:, i], pred)
    rmse = np.sqrt(mean_squared_error(y_test[:, i], pred))
    r2   = r2_score(y_test[:, i], pred)
    t    = time.time() - t0

    short = target.replace('TARGET_', '')
    print(f"  {short:<33} {t:>8.1f}s {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}")
    models[target]  = model
    results[target] = {'MAE':round(mae,4), 'RMSE':round(rmse,4), 'R2':round(r2,4)}

total_time = time.time() - t_total
avg_mae  = np.mean([v['MAE']  for v in results.values()])
avg_rmse = np.mean([v['RMSE'] for v in results.values()])
avg_r2   = np.mean([v['R2']   for v in results.values()])

print("-" * 72)
print(f"  {'AVERAGE':<33} {total_time:>8.1f}s "
      f"{avg_mae:>8.4f} {avg_rmse:>8.4f} {avg_r2:>8.4f}")

# Feature importances
importances = np.mean([models[t].feature_importances_ for t in TARGET_COLS], axis=0)
top_idx = np.argsort(importances)[::-1][:15]
print(f"\nTop 15 features:")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {FEAT_COLS[idx]:<45} {importances[idx]:.4f}")

# ── Save ──────────────────────────────────────────────────────
print(f"\nSaving to {OUTPUT_DIR}...")
pd.DataFrame(all_preds, columns=[c+'_pred' for c in TARGET_COLS]).to_csv(
    f'{OUTPUT_DIR}/model1_predictions.csv', index=False)
with open(f'{OUTPUT_DIR}/model1_summary.json', 'w') as f:
    json.dump({
        'model': 'XGBoost_Baseline',
        'train_time_sec': round(total_time,2),
        'avg_MAE': round(avg_mae,4),
        'avg_RMSE': round(avg_rmse,4),
        'avg_R2': round(avg_r2,4),
        'per_target': results,
        'top_features': [(FEAT_COLS[i], round(float(importances[i]),4)) for i in top_idx],
        'n_train': int(X_train.shape[0]),
        'n_test':  int(X_test.shape[0]),
        'n_features': int(X_train.shape[1]),
        'train_losses': [], 'val_losses': [],
    }, f, indent=2)
with open(f'{OUTPUT_DIR}/model1_xgboost.pkl', 'wb') as f:
    pickle.dump(models, f)

print("  ✅ model1_predictions.csv")
print("  ✅ model1_summary.json")
print("  ✅ model1_xgboost.pkl")
print("\n✅ Model 1 done!")

Mounted at /content/drive
Loading data...
X_train: (201984, 123)  y_train: (201984, 4)
X_test:  (135120, 123)   y_test:  (135120, 4)

Capping test features to train range...
  66 columns had out-of-range values — clipped to train range
  Verification — Window_Inactive_Sum max before: 15.261
  Verification — Window_Inactive_Sum max after:  7.549

Training XGBoost (4 models)...
Target                                    Time      MAE     RMSE       R²
------------------------------------------------------------------------
  threshold_hours                       12.9s   1.2010   2.5481 -12.4598
  threshold_inactive_min                 8.9s 143.3483 143.9753 -113.4414
  threshold_fluctuation                  5.8s   0.0137   0.0853   0.8930
  each_hour_fluctuation                  7.6s   0.0137   0.0853   0.8930
------------------------------------------------------------------------
  AVERAGE                               35.3s  36.1442  36.6735 -31.0288

Top 15 features:
   1. Active_Hour